# 9장. 회귀 분석으로 숫자 예측하기

이 Notebook의 핵심은 높은 점수를 만드는 것이 아니라 다음 순서를 지키는 것입니다.

`예측 시점 → 누수 방지 → 시간 분할 → Train TimeSeriesSplit → 모델 선택 고정 → Final Test → Baseline 비교`

**Final Test 결과를 보고 모델을 고르지 않습니다.**

## STEP 0. 실행 전 확인

Chapter09는 Chapter05의 오류 탐지용 전용 Raw가 아니라 **공통 프로젝트 `data/raw`**를 사용합니다. 먼저 프로젝트 루트에서 다음 명령을 실행해 관계 검증을 통과한 모델링 입력을 준비합니다.

```powershell
python scripts/prepare_ch09_data.py
```

이 명령은 공통 `data/raw`를 전처리하고 FK 관계를 검증한 뒤 `data/processed/*_clean.csv`를 생성합니다.

In [1]:
# STEP 0. Kernel 확인 + Chapter09 공통 모듈 import
import sys
from pathlib import Path
import pandas as pd

print(sys.executable)   # 지금 Notebook이 어떤 Python(venv)을 쓰는지 확인
print(Path.cwd())       # Notebook이 실행되는 현재 작업 폴더 확인

CURRENT_DIR = Path.cwd().resolve()

# chapter09.ipynb는 project root 바로 아래(chapter09/)에 있으므로,
# requirements.txt가 있는 폴더를 찾을 때까지 상위로 올라가며 PROJECT_ROOT를 결정한다
PROJECT_ROOT = CURRENT_DIR
for candidate in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if (candidate / 'requirements.txt').exists():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))  # src.regression을 import할 수 있도록 경로 추가

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'   # STEP0에서 prepare_ch09_data.py가 만든 폴더
REPORT_DIR = PROJECT_ROOT / 'reports'                 # Evidence/보고서가 저장될 폴더

from src.regression import (   # Chapter09 공통 회귀 로직 (Target/Feature 정의, Pipeline, 검증 함수 등)
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    TARGET_COLUMN,
    build_feature_audit,
    build_leakage_checklist,
    build_regression_dataset,
    build_regression_validation,
    build_split_summary,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    make_regression_models,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)


C:\dev\-llm-data-analysis-course\.venv\Scripts\python.exe
C:\dev\-llm-data-analysis-course\chapter09


PROJECT_ROOT: C:\dev\-llm-data-analysis-course
PROCESSED_DIR: C:\dev\-llm-data-analysis-course\data\processed


## STEP 1. 예측 문제와 예측 시점

- Target: 주문 1건의 주문 상세 금액 합계 `order_total`
- 예측 시점: 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만 주문 상세 수량·단가·금액은 모델에 제공하지 않는 시점
- 목표 계산 재료와 사후 정보, 식별자는 feature에서 제외합니다.

In [2]:
# STEP 1. 예측 문제 정의 확인 -- Target과 허용/금지 feature 점검
print('Target:', TARGET_COLUMN)                             # 예측 대상 컬럼명
print('Allowed features:', FEATURE_COLUMNS)                 # 예측 시점에 사용 가능하다고 정한 feature
print('Forbidden features:', sorted(FORBIDDEN_FEATURES))    # Target 계산 재료/사후 정보/식별자 등 금지 목록

validate_feature_columns()   # 허용 feature 안에 금지 feature가 섞여 있으면 여기서 예외 발생
print('Feature leakage gate: PASS')   # 예외 없이 여기까지 왔다면 누수 없음


Target: order_total
Allowed features: ['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city']
Forbidden features: ['avg_unit_price', 'customer_id', 'item_count', 'line_total', 'order_id', 'order_status', 'order_total', 'product_id', 'quantity', 'total_quantity', 'unit_price']
Feature leakage gate: PASS


### 답안: 1. 예측 문제 정의
- Target: `order_total` (같은 order_id의 line_total 합계)
- 예측 단위: 주문 1건
- 예측 시점: 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만, 주문 상세 수량·단가·금액은 아직 확정되지 않은 시점
- 예측 시점에 알 수 있는 정보: order_month, order_dayofweek, age, payment_method, gender, city
- 예측 시점에 알 수 없는 정보: quantity, unit_price, item_count, avg_unit_price, total_quantity, order_status, line_total, order_total 및 order_id/customer_id/product_id 등 식별자

#### 나의 해석과 판단
주문이 실제로 접수되기 전에는 그 주문에 어떤 상품이 몇 개, 얼마에 담길지 알 수 없다. 따라서 order_status나 quantity처럼 주문이 확정된 뒤에야 알 수 있는 값을 feature로 쓰면, 모델이 실제 예측 시점에는 존재하지 않는 정보로 학습하게 된다. 이번 예측 시점은 "주문이 들어오기 직전, 고객 속성과 시점 정보만 아는 상태"로 정의했다.


## STEP 2. Feature Audit을 확인합니다

In [3]:
# STEP 2. Feature Leakage Audit 실행
feature_audit = build_feature_audit()   # feature별 사용 가능 여부·역할·판단 이유를 담은 감사표 생성
display(feature_audit)                  # Notebook에 표로 출력 (reports/ch09_regression_feature_audit.csv와 동일 내용)


,column,selected,role,reason
0,order_month,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
1,order_dayofweek,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
2,age,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
3,payment_method,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
4,gender,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
5,city,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
6,avg_unit_price,False,forbidden,주문 상세에서 만든 목표 대리 변수
7,customer_id,False,forbidden,고객 식별자
8,item_count,False,forbidden,주문 상세가 확인된 뒤 계산되는 사후 집계
9,line_total,False,forbidden,목표값을 구성하는 주문 상세 금액


### 답안: 2. Feature Leakage Audit

`build_feature_audit()` 실행 결과(`reports/ch09_regression_feature_audit.csv`)는 다음과 같다.

| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
| --- | --- | --- | --- | --- | --- |
| order_month | O | X | X | 사용 | 교육용 예측 시점에 사용 가능하다고 가정 |
| order_dayofweek | O | X | X | 사용 | 교육용 예측 시점에 사용 가능하다고 가정 |
| age | O | X | X | 사용 | 교육용 예측 시점에 사용 가능하다고 가정 |
| payment_method | O | X | X | 사용 | 교육용 예측 시점에 사용 가능하다고 가정 |
| gender | O | X | X | 사용 | 교육용 예측 시점에 사용 가능하다고 가정 |
| city | O | X | X | 사용 | 교육용 예측 시점에 사용 가능하다고 가정 |
| avg_unit_price | X | Target 대리 변수 | X | 제외 | 주문 상세에서 만든 목표 대리 변수 |
| customer_id | X | X | 식별자 | 제외 | 고객 식별자 |
| item_count | X | X | 사후 집계 | 제외 | 주문 상세가 확인된 뒤 계산되는 사후 집계 |
| line_total | X | Target 계산 재료 | X | 제외 | 목표값을 구성하는 주문 상세 금액 |
| order_id | X | X | 식별자 | 제외 | 주문 식별자 |
| order_status | X | X | 사후 정보 | 제외 | 예측 시점 이후에 확정될 수 있는 사후 정보 |
| order_total | X | Target 자체 | X | 제외 | 예측 대상 자체 |
| product_id | X | X | 식별자 | 제외 | 주문 상세가 확인되어야 알 수 있는 식별자 |
| quantity | X | Target 계산 재료 | X | 제외 | 목표값 계산 재료 |
| total_quantity | X | Target 대리 변수 | X | 제외 | 주문 상세에서 만든 목표 대리 변수 |
| unit_price | X | Target 계산 재료 | X | 제외 | 목표값 계산 재료 |

- 금지 feature overlap 개수: 0 (`validate_feature_columns()` → Feature leakage gate: PASS)
- 가장 위험하다고 판단한 feature: `avg_unit_price`, `total_quantity`
- 이유: 이름만 보면 "요약 통계"처럼 보여 안전한 feature로 착각하기 쉽지만, 실제로는 주문 상세(quantity × unit_price)가 확정된 뒤에야 계산할 수 있는 **Target의 대리 변수**다. order_id/customer_id 같은 식별자는 누수라는 게 바로 보이지만, 이런 파생 통계형 변수가 오히려 놓치기 쉬운 누수다.

![Leakage 점검](images/step02_leakage.png)


### STEP 3 준비. 검증된 전처리 데이터를 불러옵니다

In [4]:
# STEP 3 준비. STEP0에서 만든 검증된 processed 데이터를 불러온다
data = load_regression_source_data(PROCESSED_DIR)   # customers/orders/order_items 등을 dict로 반환
for name, frame in data.items():
    print(name, frame.shape)   # 각 테이블의 (행, 열) 개수 확인


customers (150, 6)
orders (300, 7)
order_items (764, 6)


## STEP 3. 주문 단위 Target을 만들고 관계를 검증합니다

`line_total = quantity × unit_price`를 확인하고, 주문과 Target은 `one_to_one`, 주문과 고객은 `many_to_one` 관계가 깨지지 않는지 확인합니다.

In [5]:
# STEP 3. 주문 단위 Target 생성 + 관계 검증 (line_total 재계산, one_to_one/many_to_one 확인)
model_data = build_regression_dataset(
    customers=data['customers'],
    orders=data['orders'],
    order_items=data['order_items'],
    # 내부적으로 저장된 line_total을 quantity*unit_price로 재검증하고,
    # orders<->order_total(one_to_one), orders->customers(many_to_one) 관계도 함께 검증한다.
    # 관계가 깨지면 조용히 넘어가지 않고 여기서 바로 ValueError를 발생시킨다.
)
display(model_data.head())
print('Model rows:', len(model_data))


,order_id,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek,order_total,gender,age,city
0,238,124,2025-07-09,bank_transfer,completed,7,2,361000,M,67,대구
1,79,62,2025-07-11,bank_transfer,refunded,7,4,283000,F,63,광주
2,12,98,2025-07-12,kakao_pay,completed,7,5,444000,F,61,인천
3,169,9,2025-07-12,kakao_pay,completed,7,5,448000,M,69,서울
4,152,62,2025-07-13,naver_pay,refunded,7,6,985000,F,63,광주


Model rows: 300


### 답안: 3. Target 생성과 관계 검증
- line_total 계산식: `line_total = quantity * unit_price`
- order_total 계산식: `order_total = order_id`별 `line_total` 합계
- line_total 불일치 건수: 0건 (order_items 764행 전수 확인, `|저장된 line_total - quantity*unit_price| > 1e-6`인 행 없음)
- orders ↔ target 관계 결과: one_to_one PASS (부모 주문 없는 target 0건, target 없는 주문 0건)
- orders → customers 관계 결과: many_to_one PASS (customers에 없는 customer_id 0건)

#### 나의 해석과 판단
관계 검증 없이 그냥 merge만 하면, 주문은 있는데 target이 빠지거나(inner join이면 조용히 사라짐) target 쪽에 중복이 생겨도(outer join에 검증 없이) 티가 나지 않는다. 특히 pandas의 기본 merge는 관계가 깨져도 에러를 내지 않고 조용히 행을 늘리거나 줄여버리기 때문에, `validate="one_to_one"` / `validate="many_to_one"`처럼 관계를 명시적으로 강제해야 이런 손실·중복을 즉시 잡아낼 수 있다. 이번 데이터는 762행 order_items → 300개 order_id로 정확히 집계되고, 여기서 나온 target이 300개 주문 전부와 1:1로 맞아떨어지는 것을 직접 확인했다.


## STEP 4. 시간 순서로 Train / Final Test를 분리합니다

같은 달력 날짜를 양쪽에 나누지 않습니다. Final Test는 모델 선택이 끝날 때까지 사용하지 않습니다.

In [6]:
# STEP 4. Train / Final Test를 달력 날짜 기준 시간 순서로 분할 (무작위 분할 아님)
train_data, test_data = split_model_data_by_time(model_data, test_size=0.2)   # 같은 날짜가 양쪽에 섞이지 않게 분할
split_summary = build_split_summary(train_data, test_data)   # 분할 결과 요약(행 수, 비율, 시작/종료일)
display(split_summary)

assert train_data['order_date'].max().normalize() < test_data['order_date'].min().normalize()   # 시간 역전 방지
print('Chronological split gate: PASS')


,split,rows,ratio_pct,start_date,end_date
0,train,244,81.33,2025-07-09,2026-04-13
1,test,56,18.67,2026-04-14,2026-07-08


Chronological split gate: PASS


In [7]:
# STEP 4. Train/Final Test 각각에서 feature(X)와 target(y) 분리
X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()   # Final Test의 y_test는 STEP 9(Final Test 평가) 전까지 사용하지 않는다
print('Train:', X_train.shape, 'Final Test:', X_test.shape)


Train: (244, 6) Final Test: (56, 6)


### 답안: 4. Train / Final Test 분할
- Train 시작일: 2025-07-09
- Train 종료일: 2026-04-13
- Train 행 수: 244 (81.33%)
- Final Test 시작일: 2026-04-14
- Final Test 종료일: 2026-07-08
- Final Test 행 수: 56 (18.67%)
- 같은 달력 날짜 중복 여부: 없음 (train/test 날짜 집합 교집합 0건, Train 최대일 < Final Test 최소일 조건 True)

#### 분할 판단
이번 문제는 "미래 주문의 금액을 지금 시점에서 예측"하는 것이 목적이라, 검증도 실제 서비스 상황과 같은 시간 순서를 지켜야 의미가 있다. 무작위 분할을 쓰면 미래 주문이 Train에 섞여 들어가 모델이 사실상 답을 미리 보고 학습하는 것과 같아지고(시간적 누수), Train CV 점수가 실제 운영 성능보다 낙관적으로 나온다. 그래서 과거 244건으로 학습하고, 이후 56건(2026-04-14~2026-07-08)을 실제 운영처럼 마지막에 한 번만 열어보는 구조를 사용했다.

![시간 분할](images/step04_split.png)


## STEP 5. 전처리는 Pipeline 내부에서 수행합니다

전처리도 모델 학습의 일부다. 숫자형 feature(order_month, order_dayofweek, age)는 결측치를 median으로 대체한 뒤 StandardScaler로 표준화하고, 범주형 feature(payment_method, gender, city)는 결측치를 최빈값으로 대체한 뒤 OneHotEncoder(handle_unknown="ignore")로 인코딩한다. 전체 데이터에서 먼저 fit한 뒤 Train/Test로 나누는 순서는 쓰지 않는다 — 대신 이 전처리를 모델과 같은 Pipeline 안에 넣어, 각 Train(또는 Train Fold)에서만 fit하고 Validation/Final Test에는 transform만 적용한다. (실제 구현·검증은 바로 다음 STEP 6의 코드 셀에서 `make_regression_models()`가 만드는 Pipeline을 통해 이뤄진다.)

### 답안: 5. 전처리는 Pipeline 내부에서 수행합니다
- 숫자형 처리: `order_month, order_dayofweek, age` → SimpleImputer(median) → StandardScaler
- 범주형 처리: `payment_method, gender, city` → SimpleImputer(most_frequent) → OneHotEncoder(handle_unknown="ignore")
- 전처리 fit 범위: `ColumnTransformer`를 `Pipeline([('preprocessor', ...), ('model', ...)])`의 첫 단계로 포함시켜, Train(또는 Train CV의 각 Fold)에서만 `fit`하고 Final Test에는 `transform`만 적용
- 전체 데이터에 먼저 fit하면 안 되는 이유: 전체 데이터(Train+Test)에서 median/평균/카테고리 목록을 먼저 계산해버리면, 그 통계 안에 Final Test의 정보가 이미 섞여 들어간다. 그러면 Final Test는 더 이상 "모델이 한 번도 보지 못한 데이터"가 아니게 되어 성능이 실제보다 낙관적으로 측정된다(전처리 단계의 데이터 누수).

STEP 6에서 만든 공식 `make_regression_models()`의 전처리(`ColumnTransformer`)를 수동으로 동일하게 재구성해 Train에서만 `fit`한 뒤 비교한 결과, 두 결과가 완전히 일치함을 확인했다(`np.allclose` True) — 공식 파이프라인이 이 규칙을 정확히 지키고 있음을 직접 검증한 것이다.

## STEP 6. Baseline과 후보 모델을 준비합니다

후보는 Baseline Mean(`DummyRegressor(strategy="mean")`), Linear Regression, Random Forest 세 모델이다. Baseline은 성능이 낮은 모델이 아니라, 복잡한 모델이 실제로 추가 가치를 만드는지 판단하기 위한 기준점이다. "Random Forest가 복잡하다는 사실만으로 평균 예측보다 우수하다고 말할 수 있는가?"라는 질문에는 실제 검증 결과 없이는 답할 수 없다 — 그래서 다음 STEP에서 Train CV로 직접 비교한다.

숫자형 결측 대체·표준화와 범주형 결측 대체·One-Hot Encoding은 모델 Pipeline 안에서 각 Train Fold로만 학습됩니다.

In [8]:
# STEP 6. 전처리(ColumnTransformer)를 포함한 Pipeline 후보 모델 준비
models = make_regression_models(random_state=42)
# 각 모델은 Pipeline([('preprocessor', ColumnTransformer(...)), ('model', 실제모델)]) 구조이므로
# 아래 STEP 7(Train CV)/STEP 9(Final Test)에서 fit할 때마다 전처리도 그 Fold의 Train에서만 다시 학습된다.
print(list(models))


['Baseline Mean', 'Linear Regression', 'Random Forest']


In [9]:
# STEP 5 규칙 검증 -- STEP 6의 공식 make_preprocessor()와 동일 구조를 직접 재구성해 비교
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np

manual_numeric = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # 숫자형 결측치 -> 중앙값
    ('scaler', StandardScaler()),                    # 표준화
])
manual_categorical = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # 범주형 결측치 -> 최빈값
    ('encoder', OneHotEncoder(handle_unknown='ignore')),   # 원-핫 인코딩
])
manual_preprocessor = ColumnTransformer([
    ('numeric', manual_numeric, ['order_month', 'order_dayofweek', 'age']),
    ('categorical', manual_categorical, ['payment_method', 'gender', 'city']),
])
manual_preprocessor.fit(X_train)                       # Train에서만 fit
X_train_manual = manual_preprocessor.transform(X_train)
X_test_manual = manual_preprocessor.transform(X_test)  # Test는 transform만

official_preprocessor = models['Random Forest'].named_steps['preprocessor']
official_preprocessor.fit(X_train)
X_train_official = official_preprocessor.transform(X_train)
X_test_official = official_preprocessor.transform(X_test)

print('X_train raw shape:', X_train.shape, '-> encoded shape:', X_train_official.shape)
print('X_test raw shape:', X_test.shape, '-> encoded shape:', X_test_official.shape)
print('공식과 결과 일치 (train):', np.allclose(X_train_official, X_train_manual))
print('공식과 결과 일치 (test):', np.allclose(X_test_official, X_test_manual))


X_train raw shape: (244, 6) -> encoded shape: (244, 19)
X_test raw shape: (56, 6) -> encoded shape: (56, 19)
공식과 결과 일치 (train): True
공식과 결과 일치 (test): True


## STEP 7. Train 기간의 TimeSeriesSplit으로 후보를 비교합니다

여기까지는 Final Test의 `y_test`를 모델 선택에 사용하지 않습니다.

In [10]:
# STEP 7. Train 기간을 TimeSeriesSplit으로 나눠 후보 모델을 비교
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
    # y_test/X_test는 여기서 전혀 사용하지 않는다 -- Final Test는 아직 열어보지 않는다
)
display(cv_summary)   # 후보별 CV MAE 평균/표준편차, CV R2 평균


,model,n_splits,cv_MAE_mean,cv_MAE_std,cv_R2_mean
0,Baseline Mean,5,423004.808743,44153.043466,-0.022821
1,Random Forest,5,437352.803929,26318.793429,-0.109687
2,Linear Regression,5,461386.286905,77792.799324,-0.310517


## STEP 8. Selected Model을 Final Test 전에 고정합니다

Linear Regression과 Random Forest 중 Train CV의 평균 MAE가 더 낮은 모델을 선택합니다. **이 시점까지 Final Test 성능은 보지 않습니다.**

In [11]:
# STEP 8. Train CV 결과만 보고 비베이스라인 모델을 고정 (Final Test 성능은 아직 안 봄)
selected_model_name = select_diagnostic_model(cv_summary)   # CV MAE가 더 낮은 모델을 선택
print('Frozen selected model:', selected_model_name)


Frozen selected model: Random Forest


### 답안: 8. Train CV로 선택한 모델
- Selected Model: Random Forest
- 선택 기준: 비베이스라인 후보(Linear Regression, Random Forest) 중 Train CV의 cv_MAE_mean이 더 낮은 모델
- 근거가 된 cv_MAE_mean: Random Forest 437352.80 vs Linear Regression 461386.29 (Random Forest가 더 낮음)
- Final Test를 보기 전에 선택했는가: 예 (STEP 9에서 y_test를 처음 사용하기 전에 이 셀에서 이미 고정)


## STEP 9. 이제 Final Test에서 Baseline과 고정 모델만 평가합니다

Final Test 결과가 기대와 다르더라도 같은 Test를 보고 다른 후보로 교체하지 않습니다.

In [12]:
# STEP 9. 이제 Final Test에서 Baseline과 고정 모델만 평가 (여기서 처음 y_test 사용)
model_comparison, predictions = train_and_evaluate_models(
    models=models,
    selected_model_name=selected_model_name,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)
display(model_comparison)   # Train/Test MAE·RMSE·R2, Baseline 대비 개선율


,model,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,baseline,416266.191884,443449.648712,522189.699620,-0.001121,0.00
1,Random Forest,selected_by_train_cv,326438.478132,458819.917234,548190.446723,-0.103298,-3.47


## STEP 10. MAE, RMSE, R²를 함께 해석합니다

### 지표 읽기

- MAE: 평균적으로 얼마만큼 틀리는가
- RMSE: 큰 오차에 더 민감
- R²: 평균 예측 대비 설명력, 음수가 될 수 있음
- MAE 개선율: Dummy baseline 대비 고정 모델의 Final Test MAE 개선 정도

### 답안: 10. Final Test 해석

**결과 관찰**
- MAE: 458,819.92원 (Baseline 443,449.65원보다 큼)
- RMSE: 548,190.45 (Baseline 522,189.70보다 큼)
- R²: -0.1033 (Baseline -0.0011보다 더 낮음, 즉 평균값 예측보다도 설명력이 낮음)
- Baseline 대비 MAE 개선율: -3.47% (개선이 아니라 악화)

**나의 해석과 판단**
Frozen Model(Random Forest)은 Train MAE(326,438)가 Test MAE(458,820)보다 훨씬 낮다 — Train에서는 잘 맞추는 것처럼 보이지만 Final Test에서는 오히려 Baseline(단순 평균 예측)보다 못하다. 이는 전형적인 과적합 신호이며, Train CV 단계의 cv_R2_mean(Random Forest -0.1097)이 Baseline(-0.0228)보다 이미 더 낮았다는 것을 다시 보면, 애초에 Train CV 시점부터 경고 신호가 있었다.

**업무·분석적 의미**
지금 이 모델을 그대로 운영에 쓰면, 그냥 전체 평균 주문 금액으로 예측하는 것보다 더 나쁜 예측을 제공하게 된다. 즉 현재 feature(order_month, order_dayofweek, age, payment_method, gender, city)만으로는 주문 금액을 의미 있게 설명하지 못한다.

**현재 한계**
Train 244건 / Test 56건으로 표본이 적고, 금지된 feature(수량·단가·품목 등)를 제외하고 나면 남는 정보가 고객 속성과 주문 시점뿐이라 애초에 신호가 약할 수 있다. R²가 음수라는 사실 자체를 코드 오류로 보지 않고, 있는 그대로(성능이 낮다)로 받아들인다.

## STEP 11. 대표 오차 사례를 진단합니다

In [13]:
# STEP 11. 실제값 vs 예측값, 잔차(residual)로 오차 사례 진단
prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)
display(prediction_result.head(10))
print('주의: order_id가 포함된 이 표는 내부 진단용입니다.')   # 식별자가 있으므로 공개용 결과와 구분해야 함


,order_id,order_date,actual_order_total,predicted_order_total,residual,abs_error,model
0,51,2026-06-15,1877000,605528.715709,1.271471e+06,1.271471e+06,Random Forest
1,21,2026-06-19,1767000,756889.654706,1.010110e+06,1.010110e+06,Random Forest
2,145,2026-05-27,1863000,861784.708271,1.001215e+06,1.001215e+06,Random Forest
3,96,2026-05-11,1690000,694946.179725,9.950538e+05,9.950538e+05,Random Forest
4,183,2026-05-23,1685000,745836.316118,9.391637e+05,9.391637e+05,Random Forest
5,168,2026-06-11,10000,897415.394314,-8.874154e+05,8.874154e+05,Random Forest
6,15,2026-05-23,1677000,849179.542426,8.278205e+05,8.278205e+05,Random Forest
7,179,2026-07-08,86000,904745.854532,-8.187459e+05,8.187459e+05,Random Forest
8,282,2026-04-24,1566000,758941.292777,8.070587e+05,8.070587e+05,Random Forest
9,73,2026-06-14,72000,871636.144070,-7.996361e+05,7.996361e+05,Random Forest


주의: order_id가 포함된 이 표는 내부 진단용입니다.


### 답안: 11. 대표 오차 사례

**사례 1 (order_id 51)**
- 실제값: 1,877,000
- 예측값: 605,528.72
- residual: 1,271,471.28 (실제보다 낮게 예측)
- absolute error: 1,271,471.28
- 관찰: 이번 Final Test 중 절대오차가 가장 큰 사례이며, 실제 주문 금액이 예측보다 3배 이상 크다.
- 가능한 이유 후보(가설, 단정 아님): 고액 주문일수록 현재 남아있는 feature(시점·고객 속성)만으로는 설명이 안 되는 다른 요인(예: 상품 구성)이 금액을 크게 좌우했을 수 있다.
- 추가 확인: 이 주문의 품목 수·구성을 (허용된 feature로 재가공 가능한 범위에서) 살펴봐야 한다.

**사례 2 (order_id 168)**
- 실제값: 10,000
- 예측값: 897,415.39
- residual: -887,415.39 (실제보다 훨씬 높게 예측)
- absolute error: 887,415.39
- 관찰: 실제로는 매우 소액 주문인데 모델은 고액으로 예측했다 — 사례 1과 정반대 방향의 큰 오차.
- 가능한 이유 후보(가설, 단정 아님): 이 고객/시점의 속성 조합이 다른 고액 주문들과 비슷해 보여, 모델이 소액 주문의 신호를 구분하지 못했을 수 있다.
- 추가 확인: 같은 속성 조합(같은 달·요일·결제수단 등)을 가진 다른 주문들의 실제 금액 분포를 봐야 한다.

두 사례 모두 원인을 데이터가 직접 보여주지 않으므로, 이유는 사실이 아니라 가설로만 기록한다. 내부 진단 파일(`reports/ch09_regression_predictions_internal.csv`)에는 order_id가 포함되어 있어 공개 보고서와는 분리해서 관리한다.

## STEP 12. 자동 Validation Evidence를 확인합니다

In [14]:
# STEP 12. 지금까지의 절차가 규칙(시간분할/누수방지/모델고정 등)을 지켰는지 자동 검증
validation = build_regression_validation(
    train_data=train_data,
    test_data=test_data,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
)
display(validation)
assert validation['status'].eq('PASS').all()   # 하나라도 FAIL이면 여기서 중단


,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


### 답안: 12. 자동 Validation Evidence

| check | value | status |
| --- | --- | --- |
| forbidden_feature_overlap | 0 | PASS |
| strict_train_before_test | True | PASS |
| selected_model_exists_in_train_cv | True | PASS |
| final_test_contains_baseline | True | PASS |
| final_test_contains_frozen_selected_model | True | PASS |
| test_rows_for_r2 | 56 | PASS |

6개 항목 모두 PASS다. FAIL 항목: 없음.

## STEP 13. Evidence와 공개 범위를 구분합니다

In [15]:
# STEP 13. Evidence(자동 검증용) / 내부 진단용 / 공개 결과 파일을 구분해서 저장
checklist = build_leakage_checklist()
output_paths = save_regression_outputs(
    model_data=model_data,             # 내부용 (customer_id 등 식별자 포함)
    split_summary=split_summary,       # Evidence
    feature_audit=feature_audit,       # Evidence
    cv_summary=cv_summary,             # Evidence
    selected_model_name=selected_model_name,
    model_comparison=model_comparison, # Evidence
    prediction_result=prediction_result,  # 내부용 (order_id 포함)
    validation=validation,             # Evidence
    checklist=checklist,               # Evidence
    report_dir=REPORT_DIR,
)
for name, path in output_paths.items():
    print(name, '->', path)   # 각 결과물이 저장된 실제 경로


model_data_internal -> C:\dev\-llm-data-analysis-course\reports\ch09_regression_model_data_internal.csv
split_summary -> C:\dev\-llm-data-analysis-course\reports\ch09_regression_split_summary.csv
feature_audit -> C:\dev\-llm-data-analysis-course\reports\ch09_regression_feature_audit.csv
cv_summary -> C:\dev\-llm-data-analysis-course\reports\ch09_regression_cv_summary.csv
model_comparison -> C:\dev\-llm-data-analysis-course\reports\ch09_regression_model_comparison.csv
predictions_internal -> C:\dev\-llm-data-analysis-course\reports\ch09_regression_predictions_internal.csv
validation -> C:\dev\-llm-data-analysis-course\reports\ch09_regression_validation.csv
checklist -> C:\dev\-llm-data-analysis-course\reports\ch09_regression_checklist.csv
report -> C:\dev\-llm-data-analysis-course\reports\ch09_regression_report.md
actual_vs_predicted_figure -> C:\dev\-llm-data-analysis-course\reports\figures\ch09_actual_vs_predicted.png
residual_figure -> C:\dev\-llm-data-analysis-course\reports\figures

### 답안: 13. Evidence와 공개 범위

**공개 가능한 Evidence** (`reports/`)
- `ch09_regression_split_summary.csv`, `ch09_regression_feature_audit.csv`, `ch09_regression_cv_summary.csv`, `ch09_regression_model_comparison.csv`, `ch09_regression_validation.csv`, `ch09_regression_checklist.csv`
- 공개 가능한 보고서·그래프: `ch09_regression_report.md`, `figures/ch09_actual_vs_predicted.png`, `figures/ch09_residual_histogram.png`

**Internal로만 유지할 파일**
- `ch09_regression_model_data_internal.csv` (customer_id 등 식별자 포함)
- `ch09_regression_predictions_internal.csv` (order_id 포함)

**Internal로 구분한 이유**
두 내부 파일 모두 개별 주문/고객을 특정할 수 있는 식별자(order_id, customer_id)를 그대로 담고 있다. 모델 성능이나 검증 결과를 공유하는 목적에는 식별자가 필요 없고, 식별자가 남아 있으면 불필요하게 특정 고객·주문을 역추적할 수 있는 여지를 준다. 그래서 분석 검증용(Evidence)·내부 디버깅용(Internal)·외부 공유용(공개 보고서)을 파일 단위로 분리했다.

## STEP 14. 전체 파이프라인을 처음부터 다시 실행합니다

셀의 메모리 상태가 아니라 공통 함수가 같은 순서를 재현하는지 확인합니다.

In [16]:
# STEP 14. Notebook 셀 순서/메모리 상태가 아니라, 공통 함수(run_regression_analysis)가
# 같은 입력에서 같은 결과를 재현하는지 새 실행으로 다시 확인
result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)
print('Selected by Train CV:', result['selected_model_name'])
display(result['validation'])


Selected by Train CV: Random Forest


,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


### 답안: 14. 전체 재실행 확인
- 입력 준비 성공 여부: 예 (`python scripts/prepare_ch09_data.py` 실행 결과 FK 관계 검증 3개 항목 모두 invalid_count 0)
- 회귀 스크립트 실행 성공 여부: 예 (`python scripts/run_regression_analysis.py` 정상 종료)
- Train CV Selected Model: Random Forest
- Final Validation 전체 PASS 여부: 예 (6개 항목 모두 PASS)
- Notebook과 스크립트 결과의 일관성: 완전히 일치 (Selected Model, CV 지표, Final Test 지표, Validation 결과 전부 Notebook 실행 결과와 동일한 숫자로 재현됨 — `random_state=42`로 고정되어 있어 재현 가능)

## STEP 15. 모델 사용 가능성을 판단하고 제출합니다

낮은 성능이나 음수 R²도 숨기지 않습니다. 현재 예측 시점에 허용한 정보가 약하다면 `현재 데이터로는 모델 사용 보류`도 올바른 결론입니다.

LLM이 만든 코드도 같은 예측 시점·누수·Pipeline·Train CV·Final Test 보호 기준으로 검토합니다.

### 답안: 15. 최종 사용 판단

- [x] 현재 데이터로는 사용 보류
- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능

**판단 근거**
1. Final Test R²가 -0.1033으로 음수다 — Baseline(단순 평균 예측, R² -0.0011)보다도 설명력이 낮다.
2. Baseline 대비 MAE 개선율이 -3.47%로, Frozen Model(Random Forest)이 Baseline보다 오히려 더 나쁘게 예측한다.
3. Train MAE(326,438)와 Test MAE(458,820) 격차가 크고, Train CV 단계의 cv_R2_mean(-0.1097)도 이미 Baseline보다 낮았다 — Train 단계부터 일관되게 약한 신호였다.

**업무적 위험**
이 모델을 그대로 운영에 쓰면 "그냥 평균 주문 금액으로 예측"하는 것보다 나쁜 결과를 준다. 특히 고액/소액 주문에서 오차가 크게 벌어지는 사례(STEP 11)가 있어, 재고·마케팅 예산처럼 금액 예측에 의존하는 의사결정에 쓰면 오히려 판단을 그르칠 위험이 있다.

**현재 데이터의 한계**
Train 244건/Test 56건으로 표본이 적고, 누수 방지를 위해 제거한 feature(수량·단가·품목 등)를 빼고 나면 남는 정보가 고객 속성·주문 시점 6개뿐이라 애초에 주문 금액을 설명할 신호가 부족할 수 있다.

**다음 개선 우선순위**
1. 더 많은 과거 주문 데이터를 확보해 표본을 늘린다.
2. 예측 시점에 실제로 알 수 있는 추가 feature(예: 고객의 과거 평균 구매 금액대 — 예측 시점 이전 데이터만 사용하는 방식으로)를 누수 없이 탐색한다.
3. 지금과 다른 미래 평가 구간으로 재검증해, 이번 Final Test 결과가 일회성인지 재현되는지 확인한다.

## 완료 체크리스트 (Notebook 자체 점검)

- [ ] `python scripts/prepare_ch09_data.py`로 공통 모델링 입력을 준비했다.
- [ ] Target과 예측 시점을 설명할 수 있다.
- [ ] 금지 feature가 입력에 없음을 확인했다.
- [ ] 같은 날짜가 Train과 Final Test에 동시에 들어가지 않는다.
- [ ] 전처리는 Pipeline 내부에서 학습된다.
- [ ] Train TimeSeriesSplit으로 후보를 비교했다.
- [ ] Final Test 전에 Selected Model을 고정했다.
- [ ] Final Test에서는 Baseline과 Frozen Model만 비교했다.
- [ ] MAE, RMSE, R²를 함께 읽었다.
- [ ] 내부 식별자 결과와 공개 보고서를 구분했다.
- [ ] Validation Evidence가 모두 PASS다.

## LLM 활용 기록

### 사용 목적
Feature Leakage 검토, 회귀 파이프라인(Chapter09) 코드 실행·검증, Notebook의 STEP 번호 정리, 지표·오차 해석 초안 작성

### 사용한 프롬프트 요약
- Target은 order_total, 예측 시점 이후에만 알 수 있는 정보는 feature에서 제외
- Target 계산 재료(quantity, unit_price, line_total 등)와 식별자(order_id, customer_id, product_id) 제외
- 시간 순서로 Train/Final Test 분할 (무작위 분할 금지)
- Train TimeSeriesSplit으로만 후보 비교, Final Test는 모델 선택에 사용하지 않음
- Final Test 전에 Selected Model 고정
- Baseline(Baseline Mean)과의 비교를 항상 함께 제시

### LLM 제안 중 수정한 내용
- Notebook을 `chapter09/`에 두면 공식 코드의 project root 탐색 로직(`CURRENT_DIR.name == 'notebooks'`)이 깨지는 것을 발견해, `requirements.txt`가 있는 폴더를 찾을 때까지 상위로 올라가는 방식으로 수정
- 처음에 STEP 번호를 노트북 원래 섹션 번호(0~15)에 단순히 대응시켰다가, 실제 가이드 STEP 순서와 어긋난 것(Feature Audit의 위치, STEP5/STEP6이 뒤바뀐 것, STEP10 지표 해석이 별도 STEP으로 안 빠진 것)을 발견해 가이드 원문과 다시 대조하며 재배열
- 채팅에만 보여주고 실제 Notebook 파일에는 반영되지 않았던 코드(STEP5 수동 검증)를 실제 셀로 추가하고 재실행해 확인

### 최종 판단은 누가 했는가
분석자인 내가 Evidence(cv_summary, model_comparison, validation)를 직접 확인한 뒤 STEP 15의 최종 사용 판단("현재 데이터로는 사용 보류")을 결정함. Final Test 성능이 기대보다 낮게 나왔을 때도 다른 후보로 바꾸거나 결과를 감추지 않고 그대로 기록함.

## 최종 제출 전 확인

- [x] Kernel Restart → Run All로 전체 Notebook을 재실행해 오류 없음을 확인함 (`llm-data-analysis-course` venv, Python 3.14.6)
- [x] `python scripts/prepare_ch09_data.py`, `python scripts/run_regression_analysis.py`를 실제 CLI로도 재실행해 Notebook 결과와 동일함을 확인함
- [ ] Markdown 답안이 보이는가 — GitHub에 올린 뒤 직접 확인 필요
- [ ] 표가 정상 렌더링되는가 — GitHub에 올린 뒤 직접 확인 필요
- [x] 대표 이미지 3개 존재 확인: `images/step02_leakage.png`, `images/step04_split.png`, `images/step06_final_metrics.png`
- [x] Selected Model이 Final Test 전에 기록되어 있는가 — STEP 8 답안에서 Final Test(STEP 9) 이전에 Random Forest로 고정 기록함
- [x] Validation PASS가 확인되는가 — STEP 12 답안에서 6개 항목 모두 PASS
- [ ] **개인 PC 절대 경로가 노출되지 않는가 — 주의: STEP 0 코드 셀 출력에 `C:\\dev\\-llm-data-analysis-course\\.venv\\Scripts\\python.exe` 같은 실제 PC 경로가 그대로 찍혀 있습니다. GitHub에 올리기 전에 이 셀의 출력(Clear Output)을 지우거나, 경로를 가리는 것을 권장합니다.**
- [x] 불필요한 식별자나 개인정보가 공개되지 않는가 — order_id/customer_id가 포함된 파일은 `*_internal.csv`로 분리했고(STEP 13 답안), 공개 보고서(`ch09_regression_report.md`, 그래프)에는 식별자가 없음